![servicedesk](servicedesk.png)

CleverSupport is a company at the forefront of AI innovation, specializing in the development of AI-driven solutions to enhance customer support services. Their latest endeavor is to engineer a text classification system that can automatically categorize customer complaints. 

Your role as a data scientist involves the creation of a sophisticated machine learning model that can accurately assign complaints to specific categories, such as mortgage, credit card, money transfers, debt collection, etc.

In [15]:
from collections import Counter
import nltk, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics import Accuracy, Precision, Recall

In [16]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/repl/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [17]:
# Import data and labels
with open("words.json", 'r') as f1:
    words = json.load(f1)
with open("text.json", 'r') as f2:
    text = json.load(f2)
labels = np.load('labels.npy')

In [18]:
# Dictionaries to store the word to index mappings and vice versa
word2idx = {o:i for i,o in enumerate(words)}
idx2word = {i:o for i,o in enumerate(words)}

# Looking up the mapping dictionary and assigning the index to the respective words
for i, sentence in enumerate(text):
    text[i] = [word2idx[word] if word in word2idx else 0 for word in sentence]
    
# Defining a function that either shortens sentences or pads sentences with 0 to a fixed length
def pad_input(sentences, seq_len):
    features = np.zeros((len(sentences), seq_len),dtype=int)
    for ii, review in enumerate(sentences):
        if len(review) != 0:
            features[ii, -len(review):] = np.array(review)[:seq_len]
    return features

text = pad_input(text, 50)

In [19]:
# Splitting dataset
train_text, test_text, train_label, test_label = train_test_split(text, labels, test_size=0.2, random_state=42)

train_data = TensorDataset(torch.from_numpy(train_text), torch.from_numpy(train_label).long())
test_data = TensorDataset(torch.from_numpy(test_text), torch.from_numpy(test_label).long())

In [20]:
# Start coding here

# Define the CNN classifier
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, kernel_size, num_classes):
        super(CNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1d = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size)
        self.fc = nn.Linear(num_filters, num_classes)

    def forward(self, x):
        x = self.embedding(x)           # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)         # (batch, embed_dim, seq_len) for Conv1d
        x = F.relu(self.conv1d(x))     # (batch, num_filters, seq_len - kernel_size + 1)
        x = F.max_pool1d(x, x.shape[2]).squeeze(2)  # global max pooling → (batch, num_filters)
        x = self.fc(x)                 # (batch, num_classes)
        return x

# Hyperparameters
vocab_size  = len(words)
embed_dim   = 64
num_filters = 128
kernel_size = 3
num_classes = len(np.unique(labels))
batch_size  = 32
epochs      = 3
lr          = 1e-3

# Instantiate model, loss, optimizer
model     = CNNClassifier(vocab_size, embed_dim, num_filters, kernel_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False)

# Training loop
model.train()
for epoch in range(epochs):
    total_loss = 0
    for inputs, labels_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}")

# Inference on test_data
model.eval()
predicted = []
all_labels  = []

with torch.no_grad():
    for inputs, labels_batch in test_loader:
        outputs = model(inputs)
        preds   = torch.argmax(outputs, dim=1)
        predicted.extend(preds.tolist())
        all_labels.extend(labels_batch.tolist())

# Metrics
predictions_tensor = torch.tensor(predicted)
labels_tensor      = torch.tensor(all_labels)

acc_metric  = Accuracy(task="multiclass", num_classes=num_classes)
prec_metric = Precision(task="multiclass", num_classes=num_classes, average=None)
rec_metric  = Recall(task="multiclass", num_classes=num_classes, average=None)

accuracy  = acc_metric(predictions_tensor, labels_tensor).item()
precision = prec_metric(predictions_tensor, labels_tensor).tolist()
recall    = rec_metric(predictions_tensor, labels_tensor).tolist()

print(f"\nAccuracy : {accuracy:.4f}")
print(f"Precision: {precision}")
print(f"Recall   : {recall}")

Epoch 1/3 — Loss: 1.3174
Epoch 2/3 — Loss: 0.7629
Epoch 3/3 — Loss: 0.5741

Accuracy : 0.7560
Precision: [0.7112299203872681, 0.7719298005104065, 0.8121827244758606, 0.6320000290870667, 0.8871794939041138]
Recall   : [0.6927083134651184, 0.6947368383407593, 0.7407407164573669, 0.8229166865348816, 0.8238095045089722]
